# 02 · Khám phá dữ liệu PhoNER_COVID19 — Đức Tâm

**Issue T11** · phục vụ **BC1 mục (2)** và báo cáo mục **3.1 – 3.2** · Chạy trên máy cá nhân bằng **VS Code** (không cần GPU). Cần: `numpy`, `pandas`, `matplotlib`, và `transformers` cho phần đếm subword (cài: xem ô `%pip` ở `00_setup_tam.ipynb`).

Notebook trả lời các câu hỏi mà phần dữ liệu của báo cáo phải trả lời — và **vì sao mỗi câu hỏi quan trọng với mô hình**:

| Câu hỏi | Vì sao quan trọng |
|---|---|
| Số thực thể theo loại có khớp bài báo gốc không? | Chứng minh nhóm dùng đúng dữ liệu; phát hiện loại hiếm (JOB, TRANSPORTATION, NAME) → Macro-F1 thấp hơn Micro-F1 |
| Câu dài bao nhiêu âm tiết / từ? Bao nhiêu câu vượt 256 subword? | Quyết định `max_length`; câu bị cắt thì token cuối bị tính là sai |
| Thực thể dài bao nhiêu âm tiết? | ORGANIZATION, LOCATION dài → dễ lỗi **sai biên** (nhóm lỗi ở mục 5.4) |
| Bao nhiêu thực thể ở test chưa từng gặp trong train? | Đo mức mô hình phải *tổng quát hóa* chứ không chỉ *nhớ* |
| Có token chưa chuẩn hóa Unicode (NFC) không? | Cùng một chữ có 2 cách mã hóa → tokenizer cắt khác nhau (bài giảng Session 2) |

## Khởi tạo
Tìm thư mục gốc repo, chuyển về đó và bật `autoreload` (sửa file trong `src/` rồi chạy lại ô là dùng bản mới, không cần restart kernel).

In [ ]:
# Ô khởi tạo - chạy ĐẦU TIÊN. Notebook nằm trong notebooks/, còn code/dữ liệu/kết quả tính từ THƯ MỤC GỐC repo,
# nên ô này tìm thư mục gốc rồi chuyển về đó (đúng dù VS Code mở notebook với thư mục làm việc nào).
%load_ext autoreload
%autoreload 2

import os, sys, json, glob, shutil, time
from pathlib import Path


def tim_goc_repo():
    """Đi từ thư mục hiện tại lên trên tới khi gặp thư mục có configs/common.json và src/."""
    hien_tai = Path.cwd().resolve()
    for p in [hien_tai, *hien_tai.parents]:
        if (p / "configs" / "common.json").is_file() and (p / "src").is_dir():
            return p
    raise FileNotFoundError(
        "Không tìm thấy thư mục gốc repo (cần có configs/common.json và src/). "
        "Trong VS Code: File -> Open Folder -> chọn thư mục NLP-phoner-covid19-26, rồi mở lại notebook.")


GOC_REPO = tim_goc_repo()
os.chdir(GOC_REPO)                                    # mọi đường dẫn tương đối (configs/, data/, results/, src/) tính từ gốc repo
if str(GOC_REPO / "src") not in sys.path:
    sys.path.insert(0, str(GOC_REPO / "src"))         # để `import data`, `import train_ner`, ...
print("Thư mục gốc repo :", GOC_REPO)
print("Python (kernel)  :", sys.executable)

## Code của nhóm
`configs/common.json`, `src/data.py`, `src/eda.py` đã nằm sẵn trong repo — notebook chỉ **gọi** chúng, không ghi lại như trên Colab. Muốn sửa code thì sửa thẳng file trong `src/`.

## Tải dữ liệu PhoNER_COVID19 + kiểm tra

In [ ]:
# Dữ liệu công khai của VinAI (PhoNER_COVID19) - chỉ dùng cho nghiên cứu/học tập, KHÔNG commit (đã chặn trong .gitignore).
# Hàm này tải thẳng file zip từ GitHub (không cần cài git) và chỉ tải khi data/ chưa đủ 6 file.
from data import tai_ve_du_lieu
tai_ve_du_lieu()
print(sorted(os.listdir("data")))
%run src/data.py

## Chạy EDA (kèm đếm subword bằng 3 tokenizer thật — cần internet, ~3 phút)

In [ ]:
# %run chạy script ngay trong kernel (stdout UTF-8, thấy tiến trình trực tiếp). Dùng !python trên Windows sẽ lỗi
# UnicodeEncodeError khi in tiếng Việt ra ống dẫn (pipe). Phần --tokenizer cần internet để tải 3 tokenizer từ Hugging Face.
%run src/eda.py --tokenizer

## Xem hình

In [ ]:
from IPython.display import Image, display
for h in ["hinh_thuc_the_theo_loai.png", "hinh_do_dai.png"]:
    display(Image(filename=f"results/eda/{h}", width=900))

## Gợi ý nhận xét (Tâm tự kiểm tra lại bằng số in ra ở trên rồi viết bằng lời của mình)
1. Phân bố lệch: LOCATION và PATIENT_ID chiếm phần lớn; JOB, TRANSPORTATION, NAME rất ít → dự đoán Macro-F1 < Micro-F1.
2. Thực thể ORGANIZATION trung bình dài nhất (nhiều âm tiết) → mô hình dễ sai biên ở loại này.
3. % câu vượt 256 subword của từng tokenizer → giải thích lựa chọn `max_length = 256`.
4. Tỉ lệ thực thể test chưa gặp trong train → loại nào cao nhất (NAME, TRANSPORTATION…) thì mô hình phải dựa vào ngữ cảnh.
5. Số token chưa ở dạng NFC → lý do `data.py` chuẩn hóa NFC trước khi đưa vào tokenizer.

## Các file kết quả
Tất cả nằm sẵn trong `results/eda/` của repo — commit cùng nhánh, không cần nén hay tải về như trên Colab.

In [ ]:
for f in sorted(glob.glob("results/eda/*")):
    print(f"{os.path.getsize(f):>9,d} B  {f}")